# Excel reports through ZEMI Arsenal

The OpenAI client passes JSON Schema directly to llama.cpp, so the server constrains the structure during a single generation.

In [ ]:
from pathlib import Path
while not Path('.zemicomp').is_file():
    if Path.cwd().parent == Path.cwd(): raise FileNotFoundError('ZEMI component root')
    %cd ..

In [ ]:
import zemi
from zemi.arsenal import ArsenalSession
arsenal = ArsenalSession('@comp/playbook.toml')
zemi.arsenal.begin(arsenal, stop_before_begin=True, llama_router_mode=False)
assistant = arsenal.llamas.primary.models.qwen.assistants.report_parser
client = assistant.clients.openai.client

In [ ]:
from pydantic import BaseModel, ConfigDict, Field
class StrictModel(BaseModel): model_config = ConfigDict(extra='forbid')
class Transaction(StrictModel):
    date: str = Field(description='YYYY-MM-DD')
    article: str
    cost: float
class Report(StrictModel):
    source_file: str
    city: str
    export_date: str = Field(description='YYYY-MM-DD')
    manager: str
    transactions: list[Transaction]
class Reports(StrictModel): reports: list[Report]

In [ ]:
from markitdown import MarkItDown
from zemi import env
data_dir = env.path.comp / 'data/case01'
excel_files = [data_dir / f'Report {number}.xlsx' for number in range(1, 4)]
converter = MarkItDown(enable_plugins=False)
excel_context = '\n\n'.join(f'# File: {path.name}\n\n{converter.convert(path).text_content.strip()}' for path in excel_files)

In [ ]:
import json
task = 'Extract the reports without the Total row and without invented data.'
response = client.chat.completions.create(
    model=assistant.clients.model,
    messages=[{'role':'system','content':'Convert Excel reports into strictly structured data.'},{'role':'user','content':f'{task}\n\n{excel_context}'}],
    temperature=0.0, max_tokens=2048,
    extra_body={'json_schema': Reports.model_json_schema()},
)
result = Reports.model_validate_json(response.choices[0].message.content)
print(json.dumps(result.model_dump(mode='json'), ensure_ascii=False, indent=2))

In [ ]:
zemi.arsenal.end(arsenal, stop_after_end=True)